In [1]:
import pandas as pd
import numpy as np
import os

# 1. Define Project Directories (Update the root path if necessary)
project_root = r"C:\Users\Lenovo\Desktop\Customer Retention and Churn Analytics"
raw_data_path = os.path.join(project_root, r"Data\Raw\WA_Fn-UseC_-Telco-Customer-Churn.xls")
processed_dir = os.path.join(project_root, r"Data\Processed")

# Create the processed directory if it doesn't exist
os.makedirs(processed_dir, exist_ok=True)

# 2. Load the Dataset
print("Loading dataset...")
# Note: The file has an .xls extension but is formatted internally as a CSV
df = pd.read_csv(raw_data_path)

# 3. Data Cleaning & Type Conversion
print("Cleaning data...")
# FIX: 'TotalCharges' contains blank spaces (" ") for brand-new customers (0 tenure). 
# We must replace spaces with NaN, convert to float, and fill missing values with 0.
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'].replace(" ", np.nan))
df['TotalCharges'] = df['TotalCharges'].fillna(0)

# FIX: Simplify redundant categories ('No internet service' -> 'No') for cleaner Power BI filters
columns_to_simplify = ['OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 
                       'TechSupport', 'StreamingTV', 'StreamingMovies', 'MultipleLines']
for col in columns_to_simplify:
    df[col] = df[col].replace({'No internet service': 'No', 'No phone service': 'No'})

# FIX: Standardize column names for SQL Server (SSMS) compatibility (removes spaces/parentheses)
df.columns = [col.replace(" ", "_").replace("(", "").replace(")", "") for col in df.columns]

# 4. Feature Engineering (Data Science Analytics)
print("Engineering advanced retention features...")

# Feature A: Tenure Cohorts (Essential for Task 2 Cohort Analysis)
def assign_tenure_cohort(tenure):
    if tenure <= 12: return '0-1 Year'
    elif tenure <= 24: return '1-2 Years'
    elif tenure <= 36: return '2-3 Years'
    elif tenure <= 48: return '3-4 Years'
    elif tenure <= 60: return '4-5 Years'
    else: return '5+ Years'

df['Tenure_Cohort'] = df['tenure'].apply(assign_tenure_cohort)

# Feature B: Product Engagement (Total Add-On Services Subscribed)
# Calculates how deeply integrated the customer is into the ecosystem
addon_services = ['OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies']
df['Total_AddOn_Services'] = (df[addon_services] == 'Yes').sum(axis=1)

# Feature C: Customer Value Segmentation
# Divides customers into Low, Medium, and High value tiers based on Monthly Charges
df['Value_Segment'] = pd.qcut(df['MonthlyCharges'], q=3, labels=['Low Value', 'Medium Value', 'High Value'])

# Feature D: Churn Risk Flag (Predictive Heuristic)
# Industry rule: Month-to-month customers without tech support are the highest churn risk
df['High_Risk_Flag'] = np.where((df['Contract'] == 'Month-to-month') & (df['TechSupport'] == 'No'), 1, 0)

# Feature E: Binary Churn Column (1/0 for easy Math/SQL Aggregation)
df['Churn_Value'] = np.where(df['Churn'] == 'Yes', 1, 0)

# 5. Export Processed Data for SQL Server
processed_file_path = os.path.join(processed_dir, "Processed_Telco_Churn.csv")
df.to_csv(processed_file_path, index=False)

print(f"\nSuccess! Processed data ready for SQL Server import at:\n{processed_file_path}")
print("\n--- Final Dataset Preview ---")
print(df[['customerID', 'tenure', 'Tenure_Cohort', 'Total_AddOn_Services', 'High_Risk_Flag', 'Churn_Value']].head())

Loading dataset...
Cleaning data...
Engineering advanced retention features...

Success! Processed data ready for SQL Server import at:
C:\Users\Lenovo\Desktop\Customer Retention and Churn Analytics\Data\Processed\Processed_Telco_Churn.csv

--- Final Dataset Preview ---
   customerID  tenure Tenure_Cohort  Total_AddOn_Services  High_Risk_Flag  \
0  7590-VHVEG       1      0-1 Year                     1               1   
1  5575-GNVDE      34     2-3 Years                     2               0   
2  3668-QPYBK       2      0-1 Year                     2               1   
3  7795-CFOCW      45     3-4 Years                     3               0   
4  9237-HQITU       2      0-1 Year                     0               1   

   Churn_Value  
0            0  
1            0  
2            1  
3            0  
4            1  


In [2]:
import pandas as pd
import os

print("--- Commencing EDA & Star Schema Modeling ---")

# 1. Load the processed dataset from the previous step
processed_dir = r"C:\Users\Lenovo\Desktop\Customer Retention and Churn Analytics\Data\Processed"
df = pd.read_csv(os.path.join(processed_dir, "Processed_Telco_Churn.csv"))

# ---------------------------------------------------------
# 2. EXPLORATORY DATA ANALYSIS (EDA)
# ---------------------------------------------------------
print("\n[EDA Insights]")
total_customers = len(df)
churn_rate = (df['Churn_Value'].mean()) * 100
avg_tenure = df['tenure'].mean()

print(f"Total Customers: {total_customers}")
print(f"Overall Churn Rate: {churn_rate:.2f}%")
print(f"Average Customer Tenure: {avg_tenure:.1f} months")

print("\nChurn Rate by Contract Type:")
print(df.groupby('Contract')['Churn_Value'].mean() * 100)

print("\nChurn Rate by Tenure Cohort:")
print(df.groupby('Tenure_Cohort')['Churn_Value'].mean().sort_values() * 100)


# ---------------------------------------------------------
# 3. STAR SCHEMA PREPARATION (Fact & Dimension Tables)
# ---------------------------------------------------------
print("\n--- Generating Star Schema Tables ---")

# DIMENSION 1: Customer Demographics
dim_customer = df[['customerID', 'gender', 'SeniorCitizen', 'Partner', 'Dependents']].copy()
dim_customer_path = os.path.join(processed_dir, "Dim_Customer.csv")
dim_customer.to_csv(dim_customer_path, index=False)
print(f"Created Dim_Customer ({len(dim_customer.columns)} columns)")

# DIMENSION 2: Subscribed Services
dim_services = df[['customerID', 'PhoneService', 'MultipleLines', 'InternetService', 
                   'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 
                   'StreamingTV', 'StreamingMovies', 'Total_AddOn_Services']].copy()
dim_services_path = os.path.join(processed_dir, "Dim_Services.csv")
dim_services.to_csv(dim_services_path, index=False)
print(f"Created Dim_Services ({len(dim_services.columns)} columns)")

# DIMENSION 3: Billing & Contract Details
dim_billing = df[['customerID', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'Value_Segment']].copy()
dim_billing_path = os.path.join(processed_dir, "Dim_Billing.csv")
dim_billing.to_csv(dim_billing_path, index=False)
print(f"Created Dim_Billing ({len(dim_billing.columns)} columns)")

# FACT TABLE: Core Metrics & Churn Status
fact_churn = df[['customerID', 'tenure', 'Tenure_Cohort', 'MonthlyCharges', 
                 'TotalCharges', 'High_Risk_Flag', 'Churn', 'Churn_Value']].copy()
fact_churn_path = os.path.join(processed_dir, "Fact_Churn_Retention.csv")
fact_churn.to_csv(fact_churn_path, index=False)
print(f"Created Fact_Churn_Retention ({len(fact_churn.columns)} columns)")

print("\nSuccess! All Star Schema tables are saved to the Processed folder ready for SSMS.")

--- Commencing EDA & Star Schema Modeling ---

[EDA Insights]
Total Customers: 7043
Overall Churn Rate: 26.54%
Average Customer Tenure: 32.4 months

Churn Rate by Contract Type:
Contract
Month-to-month    42.709677
One year          11.269518
Two year           2.831858
Name: Churn_Value, dtype: float64

Churn Rate by Tenure Cohort:
Tenure_Cohort
5+ Years      6.609808
4-5 Years    14.423077
3-4 Years    19.028871
2-3 Years    21.634615
1-2 Years    28.710938
0-1 Year     47.438243
Name: Churn_Value, dtype: float64

--- Generating Star Schema Tables ---
Created Dim_Customer (5 columns)
Created Dim_Services (11 columns)
Created Dim_Billing (5 columns)
Created Fact_Churn_Retention (8 columns)

Success! All Star Schema tables are saved to the Processed folder ready for SSMS.


In [3]:
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

print("--- Commencing Next-Level Data Science Analytics ---")

# 1. Define Paths
processed_dir = r"C:\Users\Lenovo\Desktop\Customer Retention and Churn Analytics\Data\Processed"
fact_path = os.path.join(processed_dir, "Fact_Churn_Retention.csv")
raw_data_path = os.path.join(processed_dir, "Processed_Telco_Churn.csv")

df = pd.read_csv(raw_data_path)

# ---------------------------------------------------------
# A. VISUAL COHORT RETENTION HEATMAP
# ---------------------------------------------------------
print("\n[1/3] Generating Visual Cohort Retention Heatmap...")

# Create Cohort Pivot Table: Tenure Cohort vs Contract Churn Rate (%)
cohort_order = ['0-1 Year', '1-2 Years', '2-3 Years', '3-4 Years', '4-5 Years', '5+ Years']
cohort_pivot = df.pivot_table(index='Tenure_Cohort', columns='Contract', values='Churn_Value', aggfunc='mean') * 100
cohort_pivot = cohort_pivot.reindex(cohort_order)

# Render & Save Heatmap
plt.figure(figsize=(9, 5))
sns.heatmap(cohort_pivot, annot=True, fmt=".1f", cmap="Reds", cbar_kws={'label': 'Churn Rate (%)'})
plt.title("Customer Churn Rate (%) by Tenure Cohort & Contract Type", fontsize=13, fontweight='bold', pad=12)
plt.xlabel("Contract Type", fontweight='bold')
plt.ylabel("Tenure Cohort", fontweight='bold')
plt.tight_layout()

heatmap_save_path = os.path.join(processed_dir, "cohort_retention_heatmap.png")
plt.savefig(heatmap_save_path, dpi=300)
plt.close()
print(f"Success! Cohort Heatmap image saved to:\n{heatmap_save_path}")


# ---------------------------------------------------------
# B. MACHINE LEARNING CHURN PREDICTIVE MODEL
# ---------------------------------------------------------
print("\n[2/3] Training Random Forest Churn Predictor...")

# Select ML Feature Set
features = ['tenure', 'MonthlyCharges', 'TotalCharges', 'Total_AddOn_Services',
            'Contract', 'PaymentMethod', 'InternetService', 'TechSupport', 
            'OnlineSecurity', 'PaperlessBilling']

X = pd.get_dummies(df[features], drop_first=True)
y = df['Churn_Value']

# Train/Test Split & Random Forest Training
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
model = RandomForestClassifier(n_estimators=100, max_depth=8, random_state=42)
model.fit(X_train, y_train)

# Model Performance Evaluation
auc_score = roc_auc_score(y_test, model.predict_proba(X_test)[:, 1])
print(f"Model AUC-ROC Score: {auc_score:.4f} (Strong Predictive Performance)")

# Feature Importance Extraction
importances = pd.Series(model.feature_importances_, index=X.columns).sort_values(ascending=False)
print("\nTop 5 Key Drivers of Customer Churn:")
print(importances.head(5).round(4))


# ---------------------------------------------------------
# C. ENRICH FACT TABLE WITH CHURN PROBABILITY SCORES
# ---------------------------------------------------------
print("\n[3/3] Scoring Customer Base with Churn Probabilities...")

# Calculate individual churn probability for each customer
df['Churn_Probability_%'] = (model.predict_proba(X)[:, 1] * 100).round(2)

# Load existing Fact Table & Merge Predictions
fact_df = pd.read_csv(fact_path)
fact_df['Churn_Probability_%'] = df['Churn_Probability_%']

# Re-save Fact Table for SQL Server & Power BI Import
fact_df.to_csv(fact_path, index=False)
print(f"Updated Fact_Churn_Retention.csv with predictive scores at:\n{fact_path}")

print("\n--- Data Science Enhancements Complete ---")
print(fact_df[['customerID', 'tenure', 'Tenure_Cohort', 'Churn', 'Churn_Probability_%']].head())

--- Commencing Next-Level Data Science Analytics ---

[1/3] Generating Visual Cohort Retention Heatmap...
Success! Cohort Heatmap image saved to:
C:\Users\Lenovo\Desktop\Customer Retention and Churn Analytics\Data\Processed\cohort_retention_heatmap.png

[2/3] Training Random Forest Churn Predictor...
Model AUC-ROC Score: 0.8411 (Strong Predictive Performance)

Top 5 Key Drivers of Customer Churn:
tenure                            0.2361
TotalCharges                      0.1577
MonthlyCharges                    0.1242
InternetService_Fiber optic       0.1185
PaymentMethod_Electronic check    0.0788
dtype: float64

[3/3] Scoring Customer Base with Churn Probabilities...
Updated Fact_Churn_Retention.csv with predictive scores at:
C:\Users\Lenovo\Desktop\Customer Retention and Churn Analytics\Data\Processed\Fact_Churn_Retention.csv

--- Data Science Enhancements Complete ---
   customerID  tenure Tenure_Cohort Churn  Churn_Probability_%
0  7590-VHVEG       1      0-1 Year    No            

In [9]:
import os
import shutil

# Target directory and destination file
target_dir = r"C:\Users\Lenovo\Desktop\Customer Retention and Churn Analytics\Python Notebook"
os.makedirs(target_dir, exist_ok=True)

current_notebook_name = "Customer_Churn_and_Retention_EDA.ipynb" 
destination_path = os.path.join(target_dir, current_notebook_name)

try:
    shutil.copy(current_notebook_name, destination_path)
    print(f"Success! Notebook copied to:\n{destination_path}")
except shutil.SameFileError:
    print(f"Verified! Your notebook is already located at:\n{destination_path}")
except FileNotFoundError:
    print(f"Error: Could not find '{current_notebook_name}'. Make sure to save the notebook in Jupyter first.")

Verified! Your notebook is already located at:
C:\Users\Lenovo\Desktop\Customer Retention and Churn Analytics\Python Notebook\Customer_Churn_and_Retention_EDA.ipynb
